In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/audio-dataset/834608010218654.wav
/kaggle/input/audio-dataset/11992.wav
/kaggle/input/audio-dataset/834608004582894_2.wav
/kaggle/input/audio-dataset/download.wav
/kaggle/input/audio-dataset/834608004582894.wav
/kaggle/input/audio-dataset/audiomass-output2.wav
/kaggle/input/audio-dataset/834608004032583.wav
/kaggle/input/audio-dataset/16000.wav
/kaggle/input/audio-dataset/834608007007257.wav
/kaggle/input/audio-dataset/audiomass-output.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/database.yml
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004576951.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004574706.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004579439.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004581336.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004359857.wav
/kaggle/input/audio-dataset/cs_dataset/cs_dataset/audio/834608004572266.wav
/kaggle/input/audio-dat

In [2]:
!pip install librosa

In [3]:
import librosa
file_path = "/kaggle/input/audio-dataset/11992.wav"

# Load audio
signal, sr = librosa.load(file_path, sr=16000, mono=True)
print("Sample rate:", sr)
print("Duration (sec):", len(signal)/sr)


Sample rate: 16000
Duration (sec): 47.990625


In [21]:
frame_length = int(0.025 * sr)   # 400 samples with 25 ms
hop_length   = int(0.010 * sr)   # 160 samples with 10 ms


In [22]:
frame_length_lib_safe=int(0.026*sr)

In [23]:
frame_length_lib_safe

416

In [11]:
energy = np.array([
    np.sum(np.square(signal[i:i+frame_length]))
    for i in range(0, len(signal)-frame_length, hop_length)
])


In [12]:
energy

array([2.3147935e-05, 3.2031818e-05, 3.1024520e-05, ..., 1.4203212e-03,
       9.7218831e-04, 6.6134415e-04], dtype=float32)

In [13]:
zcr = librosa.feature.zero_crossing_rate(
    signal,
    frame_length=frame_length,
    hop_length=hop_length
)[0]


In [14]:
zcr

array([0.205 , 0.3275, 0.3   , ..., 0.095 , 0.09  , 0.0325])

In [15]:
def spectral_entropy(signal, sr, frame_length, hop_length):
    entropies = []

    for i in range(0, len(signal)-frame_length, hop_length):
        frame = signal[i:i+frame_length]

        spectrum = np.abs(np.fft.fft(frame))**2
        spectrum = spectrum[:len(spectrum)//2]

        prob = spectrum / np.sum(spectrum + 1e-12)
        entropy = -np.sum(prob * np.log2(prob + 1e-12))

        entropies.append(entropy)

    return np.array(entropies)

entropy = spectral_entropy(signal, sr, frame_length, hop_length)


In [19]:
pitch = librosa.yin(
    signal,
    fmin=80,     # instead of 50
    fmax=400,
    sr=sr,
    frame_length=frame_length,
    hop_length=hop_length
)


/tmp/ipykernel_55/999778284.py:1: UserWarning: With fmin=80.000, sr=16000 and frame_length=400, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=80.000 or frame_length=401.
  pitch = librosa.yin(


In [24]:
pitch_safe=librosa.yin(
    signal,
    fmin=80,
    fmax=416,
    sr=sr,
    frame_length=frame_length_lib_safe,
    hop_length=hop_length
)

In [27]:
mfcc = librosa.feature.mfcc(
    y=signal,
    sr=sr,
    n_mfcc=13,
    n_fft=frame_length,
    hop_length=hop_length
)

mfcc = mfcc.T   # shape: (frames, 13)
mfcc.shape

(4800, 13)

In [36]:
spectral_flux = librosa.onset.onset_strength(
    y=signal,
    sr=sr,
    hop_length=hop_length,
    detrend=False

)
spectral_flux

array([0.        , 0.        , 0.        , ..., 0.13562714, 0.08411439,
       0.08230996], dtype=float32)

In [37]:
spectral_flux.shape

(4800,)

In [38]:
min_len = min(len(energy), len(zcr), len(entropy),
              len(pitch), len(mfcc), len(spectral_flux))

features = np.column_stack([
    energy[:min_len],
    zcr[:min_len],
    entropy[:min_len],
    pitch[:min_len],
    spectral_flux[:min_len],
    mfcc[:min_len]
])


In [42]:
features

array([[ 2.31479353e-05,  2.05000000e-01,  5.72364569e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 3.20318177e-05,  3.27500000e-01,  5.89709854e+00, ...,
        -1.44924033e+00, -1.03186011e+00,  1.00136673e+00],
       [ 3.10245196e-05,  3.00000000e-01,  5.68116331e+00, ...,
         1.38676524e-01,  6.73776269e-01,  1.11935496e+00],
       ...,
       [ 1.42032118e-03,  5.25000000e-02,  2.46304297e+00, ...,
        -1.32534819e+01, -7.06669044e+00, -3.22770905e+00],
       [ 9.72188311e-04,  5.25000000e-02,  4.11847639e+00, ...,
        -1.43755236e+01, -6.29311848e+00, -8.07653236e+00],
       [ 6.61344151e-04,  7.75000000e-02,  4.20738268e+00, ...,
        -1.59377651e+01, -1.00228500e+01, -8.92117977e+00]])